# 20.1 照片、打字與說話，怎麼交給同一位助理？


假設你先打字說「接下來請用繁體中文回答」，再附一張照片問「招牌寫什麼」。下一輪改用說話提問時，剛才的照片與語言要求仍然有用。我們希望換的是提問方式，不是每按一個按鈕，就換成一位忘記前文的助理。

先讀過[7.1的對話角色](https://birdhackor.github.io/tiny-perceptron-vlm/7.1.html)與[12.14的共同聊天入口](https://birdhackor.github.io/tiny-perceptron-vlm/12.14.html)，會更容易理解這張路線圖。打字直接成為這一輪的使用者訊息；錄音先交給語音辨識器，轉成它認出的文字，再接進同一份對話。語音辨識常簡稱 ASR，可以把它想成先替你聽寫的同學。聽寫完成的文字叫逐字稿，還不是回答。

圖片另提供畫面線索，和對話一起交給圖文聊天模型。底座是我們接手時已訓練好的權重；本課另練一小份候選修正，像原書旁的修改筆記。修正要和底座一起使用，也可以關閉修正只用底座。兩種配置都能接收相同輸入，最後由驗證決定交付哪一種，詳見[20.8](https://birdhackor.github.io/tiny-perceptron-vlm/20.8.html)。聲音辨識則是前面另一站。

下圖在聊天核心裡標出「可選本課微調」。無論是否加上這份修正，打字、圖片與語音逐字稿仍會交給同一位助理。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural_shared_chat.svg")))

先用手寫的兩份訊息檢查會合點：


In [ ]:
history = [{"role": "user", "content": "接下來請用繁體中文回答。"}]
typed_question = "請讀出照片裡的招牌。"
recognized_question = "請讀出照片裡的招牌。"
typed_chat = history + [{"role": "user", "content": typed_question}]
spoken_chat = history + [{"role": "user", "content": recognized_question}]
print("聊天模型收到相同對話", typed_chat == spoken_chat)
print("這輪問題", spoken_chat[-1]["content"])

`history`是此前的訊息清單。`+`建立新的清單，把這輪問題接在後面；`[-1]`取最後一筆。這裡兩個問題由我們寫成相同，所以第一行是True，第二行是招牌問題。程式只檢查對話格式，尚未讀圖或辨識錄音。實際提問還要使用相同照片，才能比較兩條路。

真正說話時，辨識器可能漏字。因此介面會先顯示逐字稿，讓你看見助理將收到什麼，再送出問題。想重新開始才清除歷史與照片；想追問同一張圖，就保留同一段對話。這份成品先用文字回答，朗讀回覆需要另外的語音合成能力。

練習把`recognized_question`改成「請介紹這張照片」。結果應變False，因為下一站已收到不同問題。記住這個會合點，下一節就可以實際打字、附圖與說話；[20.12](https://birdhackor.github.io/tiny-perceptron-vlm/20.12.html)再教你定位哪一站造成錯誤。
